<a href="https://colab.research.google.com/github/aayushcs007/BIS_LAB-1BF24CS007-/blob/main/ACO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import math
import random

# ============================================================
# IMPROVED ACO FOR REAL-TIME VEHICLE ROUTING
# Inspired by the selected 2016 research paper
# ============================================================

random.seed(42)

# ------------------- PARAMETERS -----------------------------

NUM_VEHICLES = 3
NUM_ANTS = 30
NUM_ITERATIONS = 150

ALPHA = 1.0
BETA = 2.0
RHO = 0.1
Q = 100.0

SPEED = 30.0             # km/hour
SERVICE_TIME = 2.0       # minutes per customer
RESPONSE_WEIGHT = 0.15

INITIAL_PHEROMONE = 1.0

# ------------------- LOCATIONS ------------------------------

# Location 0 is the depot.
locations = {
    0: (0, 0),
    1: (2, 3),
    2: (5, 4),
    3: (6, 1),
    4: (1, 5),
    5: (4, 7),
    6: (8, 5)
}

names = {
    0: "Depot",
    1: "Customer A",
    2: "Customer B",
    3: "Customer C",
    4: "Customer D",
    5: "Customer E",
    6: "Customer F"
}

# Request arrival times in minutes.
initial_requests = {
    1: 0,
    2: 0,
    3: 0,
    4: 5,
    5: 10
}

# ------------------- DISTANCES -----------------------------

N = len(locations)

def distance(i, j):
    x1, y1 = locations[i]
    x2, y2 = locations[j]

    return math.sqrt(
        (x1 - x2) ** 2 +
        (y1 - y2) ** 2
    )


dist = [
    [distance(i, j) for j in range(N)]
    for i in range(N)
]


def initialize_pheromone():
    return [
        [INITIAL_PHEROMONE for _ in range(N)]
        for _ in range(N)
    ]


# ------------------- OBJECTIVE FUNCTION --------------------

def evaluate(routes, requests):
    """
    Calculate:
    1. Total distance
    2. Total customer response time
    3. Average response time
    4. Combined objective cost
    """

    total_distance = 0.0
    total_response = 0.0
    responses = {}

    for route in routes:

        current = 0
        current_time = 0.0

        for customer in route:

            travel_distance = dist[current][customer]

            travel_time = (
                travel_distance / SPEED
            ) * 60.0

            arrival_time = current_time + travel_time

            # A vehicle cannot serve a customer
            # before the request has arrived.
            service_start = max(
                arrival_time,
                requests[customer]
            )

            response_time = (
                service_start - requests[customer]
            )

            total_distance += travel_distance
            total_response += response_time

            responses[customer] = response_time

            current_time = (
                service_start + SERVICE_TIME
            )

            current = customer

        # Return the vehicle to the depot.
        total_distance += dist[current][0]

    average_response = (
        total_response / len(requests)
        if requests else 0.0
    )

    # Weighted multi-objective function.
    cost = (
        total_distance
        + RESPONSE_WEIGHT * total_response
    )

    return {
        "distance": total_distance,
        "total_response": total_response,
        "average_response": average_response,
        "cost": cost,
        "responses": responses
    }


# ------------------- BUILD A SOLUTION ----------------------

def construct_solution(customers, requests, pheromone):

    routes = [[] for _ in range(NUM_VEHICLES)]

    current_nodes = [0] * NUM_VEHICLES
    current_times = [0.0] * NUM_VEHICLES

    unvisited = set(customers)

    while unvisited:

        # Encourage balanced allocation.
        route_sizes = [len(route) for route in routes]
        minimum_size = min(route_sizes)

        eligible_vehicles = [
            v for v in range(NUM_VEHICLES)
            if len(routes[v]) == minimum_size
        ]

        choices = []
        weights = []

        for vehicle in eligible_vehicles:

            current = current_nodes[vehicle]
            current_time = current_times[vehicle]

            for customer in unvisited:

                d = dist[current][customer]

                travel_time = (d / SPEED) * 60.0

                arrival_time = current_time + travel_time

                service_start = max(
                    arrival_time,
                    requests[customer]
                )

                response = (
                    service_start - requests[customer]
                )

                # Distance heuristic.
                eta = 1.0 / (d + 1e-9)

                # Attractiveness of this move.
                attractiveness = (
                    pheromone[current][customer] ** ALPHA
                ) * (
                    eta ** BETA
                ) / (
                    1.0 + RESPONSE_WEIGHT * response
                )

                choices.append((vehicle, customer))
                weights.append(attractiveness)

        # Probabilistic selection.
        selected_index = random.choices(
            range(len(choices)),
            weights=weights,
            k=1
        )[0]

        vehicle, customer = choices[selected_index]

        current = current_nodes[vehicle]

        routes[vehicle].append(customer)

        d = dist[current][customer]
        travel_time = (d / SPEED) * 60.0

        arrival_time = current_times[vehicle] + travel_time

        service_start = max(
            arrival_time,
            requests[customer]
        )

        current_times[vehicle] = (
            service_start + SERVICE_TIME
        )

        current_nodes[vehicle] = customer

        unvisited.remove(customer)

        # Local pheromone update.
        pheromone[current][customer] = (
            (1 - RHO) * pheromone[current][customer]
            + RHO * INITIAL_PHEROMONE
        )

        pheromone[customer][current] = (
            pheromone[current][customer]
        )

    return routes


# ------------------- PHEROMONE UPDATE ----------------------

def update_pheromone(pheromone, best_routes, best_cost):

    # Evaporation.
    for i in range(N):
        for j in range(N):

            pheromone[i][j] *= (1 - RHO)

    # Reinforce the best solution.
    deposit = Q / (best_cost + 1e-9)

    for route in best_routes:

        current = 0

        for customer in route:

            pheromone[current][customer] += deposit

            pheromone[customer][current] = (
                pheromone[current][customer]
            )

            current = customer

        # Return edge.
        pheromone[current][0] += deposit

        pheromone[0][current] = pheromone[current][0]


# ------------------- ACO ALGORITHM -------------------------

def run_aco(requests, experiment_name):

    customers = sorted(requests.keys())

    pheromone = initialize_pheromone()

    best_routes = None
    best_metrics = None

    history = []

    print("\n" + "=" * 65)
    print(experiment_name)
    print("=" * 65)

    for iteration in range(1, NUM_ITERATIONS + 1):

        iteration_best_routes = None
        iteration_best_metrics = None

        for _ in range(NUM_ANTS):

            routes = construct_solution(
                customers,
                requests,
                pheromone
            )

            metrics = evaluate(routes, requests)

            if (
                iteration_best_metrics is None
                or metrics["cost"]
                < iteration_best_metrics["cost"]
            ):

                iteration_best_routes = [
                    route.copy() for route in routes
                ]

                iteration_best_metrics = metrics

        # Preserve the best solution found so far.
        if (
            best_metrics is None
            or iteration_best_metrics["cost"]
            < best_metrics["cost"]
        ):

            best_routes = [
                route.copy()
                for route in iteration_best_routes
            ]

            best_metrics = iteration_best_metrics.copy()

        # Update pheromones using the iteration's best solution.
        update_pheromone(
            pheromone,
            iteration_best_routes,
            iteration_best_metrics["cost"]
        )

        history.append(best_metrics["cost"])

        if iteration % 30 == 0:

            print(
                f"Iteration {iteration:3d}"
                f" | Best distance: "
                f"{best_metrics['distance']:.2f} km"
                f" | Total response: "
                f"{best_metrics['total_response']:.2f} min"
                f" | Cost: {best_metrics['cost']:.2f}"
            )

    print("\nBEST ROUTES")

    for v, route in enumerate(best_routes):

        route_names = ["Depot"]

        route_names.extend(
            names[c] for c in route
        )

        route_names.append("Depot")

        print(
            f"Vehicle {v + 1}: "
            + " -> ".join(route_names)
        )

    print("\nACO PERFORMANCE")

    print(
        f"Total distance: "
        f"{best_metrics['distance']:.2f} km"
    )

    print(
        f"Total response time: "
        f"{best_metrics['total_response']:.2f} minutes"
    )

    print(
        f"Average response time: "
        f"{best_metrics['average_response']:.2f} minutes"
    )

    print(
        f"Combined objective cost: "
        f"{best_metrics['cost']:.4f}"
    )

    print("\nCUSTOMER RESPONSE TIMES")

    for customer in customers:

        print(
            f"{names[customer]}: "
            f"{best_metrics['responses'][customer]:.2f} minutes"
        )

    return best_routes, best_metrics, history


# ------------------- BASELINE ALGORITHM --------------------

def run_nearest_neighbor(requests):

    customers = set(requests.keys())

    routes = [[] for _ in range(NUM_VEHICLES)]

    current_nodes = [0] * NUM_VEHICLES
    current_times = [0.0] * NUM_VEHICLES

    while customers:

        sizes = [len(route) for route in routes]
        minimum_size = min(sizes)

        eligible = [
            v for v in range(NUM_VEHICLES)
            if len(routes[v]) == minimum_size
        ]

        best_choice = None
        best_score = float("inf")

        for vehicle in eligible:

            current = current_nodes[vehicle]

            for customer in customers:

                d = dist[current][customer]

                arrival_time = (
                    current_times[vehicle]
                    + (d / SPEED) * 60.0
                )

                service_start = max(
                    arrival_time,
                    requests[customer]
                )

                response = (
                    service_start - requests[customer]
                )

                score = (
                    d + RESPONSE_WEIGHT * response
                )

                if score < best_score:

                    best_score = score

                    best_choice = (
                        vehicle,
                        customer
                    )

        vehicle, customer = best_choice

        current = current_nodes[vehicle]

        routes[vehicle].append(customer)

        d = dist[current][customer]

        arrival_time = (
            current_times[vehicle]
            + (d / SPEED) * 60.0
        )

        service_start = max(
            arrival_time,
            requests[customer]
        )

        current_times[vehicle] = (
            service_start + SERVICE_TIME
        )

        current_nodes[vehicle] = customer

        customers.remove(customer)

    return routes, evaluate(routes, requests)


# ------------------- COMPARISON ----------------------------

def compare_results(aco_metrics, baseline_metrics):

    print("\n" + "=" * 65)
    print("ACO VS NEAREST-NEIGHBOR BASELINE")
    print("=" * 65)

    print(
        f"{'Metric':<28}"
        f"{'ACO':>16}"
        f"{'Baseline':>18}"
    )

    print("-" * 62)

    print(
        f"{'Distance (km)':<28}"
        f"{aco_metrics['distance']:>16.2f}"
        f"{baseline_metrics['distance']:>18.2f}"
    )

    print(
        f"{'Total response (min)':<28}"
        f"{aco_metrics['total_response']:>16.2f}"
        f"{baseline_metrics['total_response']:>18.2f}"
    )

    print(
        f"{'Average response (min)':<28}"
        f"{aco_metrics['average_response']:>16.2f}"
        f"{baseline_metrics['average_response']:>18.2f}"
    )

    print(
        f"{'Combined objective cost':<28}"
        f"{aco_metrics['cost']:>16.2f}"
        f"{baseline_metrics['cost']:>18.2f}"
    )

    if aco_metrics["cost"] < baseline_metrics["cost"]:

        print("\nACO achieved a lower combined objective cost.")

    elif aco_metrics["cost"] > baseline_metrics["cost"]:

        print(
            "\nThe baseline achieved a lower combined objective cost."
        )

    else:

        print("\nBoth methods achieved the same combined cost.")


# ------------------- MAIN PROGRAM --------------------------

if __name__ == "__main__":

    # Experiment 1: Five customer requests.
    aco_routes_1, aco_metrics_1, history_1 = run_aco(
        initial_requests,
        "EXPERIMENT 1: INITIAL REQUESTS"
    )

    baseline_routes_1, baseline_metrics_1 = (
        run_nearest_neighbor(initial_requests)
    )

    compare_results(
        aco_metrics_1,
        baseline_metrics_1
    )

    # Experiment 2: A new request arrives at minute 15.
    updated_requests = initial_requests.copy()

    updated_requests[6] = 15

    print("\nNEW REQUEST: Customer F at minute 15")

    aco_routes_2, aco_metrics_2, history_2 = run_aco(
        updated_requests,
        "EXPERIMENT 2: UPDATED CUSTOMER REQUESTS"
    )

    baseline_routes_2, baseline_metrics_2 = (
        run_nearest_neighbor(updated_requests)
    )

    compare_results(
        aco_metrics_2,
        baseline_metrics_2
    )

    # ---------------- FINAL COMPARISON ---------------------

    print("\n" + "=" * 65)
    print("EFFECT OF ADDING A NEW CUSTOMER")
    print("=" * 65)

    print(
        f"Distance change: "
        f"{aco_metrics_2['distance'] - aco_metrics_1['distance']:+.2f} km"
    )

    print(
        f"Response-time change: "
        f"{aco_metrics_2['total_response'] - aco_metrics_1['total_response']:+.2f} minutes"
    )

    print(
        f"Average response-time change: "
        f"{aco_metrics_2['average_response'] - aco_metrics_1['average_response']:+.2f} minutes"
    )

    print("\nSimulation completed successfully.")


EXPERIMENT 1: INITIAL REQUESTS
Iteration  30 | Best distance: 39.63 km | Total response: 54.47 min | Cost: 47.80
Iteration  60 | Best distance: 39.63 km | Total response: 54.47 min | Cost: 47.80
Iteration  90 | Best distance: 39.63 km | Total response: 54.47 min | Cost: 47.80
Iteration 120 | Best distance: 39.63 km | Total response: 54.47 min | Cost: 47.80
Iteration 150 | Best distance: 39.63 km | Total response: 54.47 min | Cost: 47.80

BEST ROUTES
Vehicle 1: Depot -> Customer A -> Depot
Vehicle 2: Depot -> Customer D -> Customer E -> Depot
Vehicle 3: Depot -> Customer C -> Customer B -> Depot

ACO PERFORMANCE
Total distance: 39.63 km
Total response time: 54.47 minutes
Average response time: 10.89 minutes
Combined objective cost: 47.7972

CUSTOMER RESPONSE TIMES
Customer A: 7.21 minutes
Customer B: 20.49 minutes
Customer C: 12.17 minutes
Customer D: 5.20 minutes
Customer E: 9.41 minutes

ACO VS NEAREST-NEIGHBOR BASELINE
Metric                                   ACO          Baseline
-